# Approach:

```
[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1)


y = [0110] (dim = 4)
x = [______] (dim = 6), needs to be found

M = 4 columns by 6 rows

need to solve: Mx = y as minimsation problem for y

0001 (3)        x1
0101 (1,3)      x2             0
0010 (2)        x3             1 (y is of dimension 4)
0011 (2,3)   x  x4    mod 2 =  1            
1010 (0,2)      x5             0
1100 (0,1)      x6           

actually no it should be

000011
010001
001110
110100

for the dimension to work out. rest is the same as above
```


## Part 1

We can hardcode it due to the modulo, a button can be pressed once or not. Pressing it twice is the same as none etc. So the solution space is tiny: 2**6 e.g. for the first one

## Part 2

Using scipy linprog which optimises https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html

```
        c @ x

    - such that ::

        A_ub @ x <= b_ub
        A_eq @ x == b_eq
        lb <= x <= ub
```

So we want to mimise sum(x), means we want to minimise c @ x with c = [1,1,1,1,...]  (lenght of x) as then c @ x == sum(x)

`A_eq @ x == b_eq` 


In [111]:
import numpy as np
import itertools
import scipy

p1 = 0
p2 = 0

for line in open("10").readlines():
    sol, *buttons, jolts = line.split(" ")
    y = np.array([1 if c == '#' else 0 for c in sol[1:-1]]) # light vector dim n
    n = len(y)
    buttons = [[int(a) for a in b[1:-1].split(',')] for b in buttons]
    m = len(buttons)
    B = np.array([[1 if a in b else 0 for a in range(n)] for b in buttons]).T  # button matrix n x m

    # part 1
    sorted_combinations = sorted(itertools.product([0,1], repeat=m), key = sum)
    for x in [np.array(i) for i in sorted_combinations]:
        if np.all((B @ x) % 2 == y): break
    p1 += sum(x)

    # part 2
    y = np.array([int(a) for a in jolts.strip()[1:-1].split(",")])

    c = np.ones(m)

    res = scipy.optimize.linprog(c=c, A_eq=B, b_eq=y, integrality=1)  # integrality: 0 is continuous, 1 means integer
    p2 += int(res.fun)
    
    
print(p1)
print(p2)

385
16757


# Drafts

# Day 10 2025

`[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1)`

0x + 0y + 0z + 1v = 0
0x + 1y + 0z + 1v = 1
0x + 0y + 1z + 0v = 1
0x + 0y + 1z + 1v = 0


Solve the system of equations: x0 + 2 * x1 = 1 and 3 * x0 + 5 * x1 = 2:

import numpy as np
a = np.array([[1, 2], [3, 5]])
b = np.array([1, 2])
x = np.linalg.solve(a, b)
x
array([-1.,  1.])

f(x) = {
    1x4 = 0
}



In [112]:
import numpy as np

# np.linalg.solve(
#     np.array([[0,0,0,1], [0,1,0,1], [0,0,1,0], [0,0,1,1]]),
#     np.array([0,0,1,1])
# )

np.linalg.solve(
    np.array([[0,0,1], [1,0,1], [0,1,0]]),
    np.array([1,1,0])
)

array([0., 0., 1.])

`[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1)`

In [113]:
def f(x):
    return 

# f(1, 1, 1, 0, 0, 0) = 3
# f(0, 1, 0, 1, 0, 2) = 4
# f(1, 0, 1, 1, 1, 1) = 5

# solution we expect: 1, 1, 1, 0, 0,0
# alernative: 

# button = vector bv
# e.g. button (3) = [0,0,0,1]
# button (0,2) = [1,0,1,0]

# state = vector sv
# [.##.] = [0,1,1,0]
# initial = [0,0,0,0]
# we transition into next state with button press
# sv0 x bv0 = sv1
# but it is not vector multiplication. instead, it is 
# addition + modulo
# (0000 + 0001)mod2 = 0001
# (0001 + 0001)mod2 = 0000
# it is not binary, because there is no flow, they are all separate

# the solutoin is a vector, but in a different vector space
# the dim != indicators, but dim == buttons
# so y = solution => y = f(0, 1, 0, 1, 0, 2) = 4
# we are looking to minimise the amount of buttons to arrive at the solution
# f(x) = sum(x)
# so f is very easy, but we have to enforce lots of constraints and the logic

# can we encapsoluate the whole logic as a constraint?
# we have to map x0,x1 into the button space